# Iris Flower Clustering Project

**Week 3 · Unsupervised Learning + Model Evaluation**

This notebook applies K-Means with **k = 3** to the provided Iris dataset, scales the measurements, reduces them to two principal components for plotting, and compares the discovered clusters with the known species labels. Since cluster IDs are arbitrary, the comparison uses the Adjusted Rand Index (ARI), which does not require cluster IDs to match species numbers.

**Dataset:** `Iris.csv` in this folder (150 rows; four measurements and a species label). The species column is held out from model training.

In [ ]:
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, confusion_matrix, silhouette_score
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path('Iris.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('week 3') / 'Iris.csv'  # supports running from the parent folder
df = pd.read_csv(DATA_PATH)
features = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
X = df[features]
y = df['Species']
print(f'Dataset: {len(df)} flowers, {len(features)} measurements')
print('Species counts:\n', y.value_counts().to_string())
df.head()

## 1. Scale measurements

K-Means uses distances, so measurements with larger numeric ranges can dominate. Standardizing each feature gives it mean 0 and standard deviation 1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 2. Choose k with the elbow method

The inertia curve measures within-cluster squared distances. Look for where adding another cluster gives diminishing improvement. We use the assignment's specified **k = 3** below; the elbow is a diagnostic, not proof of the true number of species.

In [ ]:
inertias = []
ks = range(1, 10)
for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)
plt.figure(figsize=(7, 4))
plt.plot(list(ks), inertias, marker='o')
plt.xticks(list(ks))
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow method for Iris')
plt.grid(alpha=.25)
plt.show()

## 3. Fit K-Means (k = 3) and apply PCA

PCA is fitted to the scaled measurements. The first two principal components retain the largest possible share of variance in a 2D projection. Cluster fitting still uses all four scaled measurements; PCA is for dimensionality reduction and visualization.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(X_scaled)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f'Variance retained by 2 components: {pca.explained_variance_ratio_.sum():.1%}')
print('Cluster sizes:', pd.Series(cluster_ids).value_counts().sort_index().to_dict())
print(f'Silhouette score (scaled 4D data): {silhouette_score(X_scaled, cluster_ids):.3f}')

## 4. Visualize clusters and known species

The left plot colors the PCA projection by K-Means cluster. The right plot colors the same projection by the known species labels. Cluster numbers have no intrinsic meaning, so compare the grouping patterns rather than matching color names.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, values, title, legend_title in [(axes[0], cluster_ids, 'K-Means clusters (k=3)', 'Cluster'), (axes[1], y, 'True Iris species', 'Species')]:
    points = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=pd.Categorical(values).codes, cmap='viridis', s=38, alpha=.8)
    ax.set(title=title, xlabel='Principal component 1', ylabel='Principal component 2')
    ax.grid(alpha=.2)
    handles, _ = points.legend_elements()
    labels = sorted(pd.unique(values), key=str)
    ax.legend(handles, labels, title=legend_title)
plt.show()

## 5. Compare clusters with true labels

The contingency table shows how the arbitrary cluster IDs distribute across species. The Adjusted Rand Index (ARI) measures agreement while accounting for chance: 1 means identical grouping, about 0 means chance-level agreement, and it can be negative for worse-than-chance agreement.

In [ ]:
comparison = pd.crosstab(pd.Series(y, name='True species'), pd.Series(cluster_ids, name='K-Means cluster'))
display(comparison)
ari = adjusted_rand_score(y, cluster_ids)
print(f'Adjusted Rand Index: {ari:.3f}')

### Interpretation

Setosa is typically cleanly separated by its measurements, while versicolor and virginica overlap more. K-Means is unsupervised: the species labels were not used to fit it. ARI and the table are used only afterward to evaluate how the discovered groups align with known species.

## 6. Save and reload the model

Save the fitted scaler, PCA projection, and K-Means model together so future data can be transformed and clustered with the same learned parameters. The output file is written beside this notebook/script.

In [ ]:
ARTIFACT_PATH = Path('iris_kmeans_model.joblib')
if not ARTIFACT_PATH.parent.exists():
    ARTIFACT_PATH = Path('week 3') / ARTIFACT_PATH
joblib.dump({'scaler': scaler, 'pca': pca, 'kmeans': kmeans, 'features': features}, ARTIFACT_PATH)
loaded = joblib.load(ARTIFACT_PATH)
example_scaled = loaded['scaler'].transform(X.iloc[:5])
print('Saved:', ARTIFACT_PATH.resolve())
print('Reloaded model sample predictions:', loaded['kmeans'].predict(example_scaled))